# 12 Final test results (Step 19)

The test block (the last 10% by time, 59,054 rows) is scored **once**, after the group has confirmed the experiment
freeze. This notebook trains nothing: it reads the test rows of `results/log.csv` and the saved test scores in
`results/preds/`, and builds the tables for the report.

**How the test rows are made** (`src/final_test.py`, one model per process, each person runs their own):

```
.venv/bin/python -u -m src.final_test --model catboost --freeze-confirmed
```

For each of the model's 12 cells (G0–G3 × seeds 0, 1, 2), the settings validated in Step 16 are refitted on the
train block exactly as in the validation run, the refit must reproduce the logged validation PR-AUC, and the test
block is scored with the threshold stored at validation. Nothing is tuned or re-thresholded on the test block.

The **segment blend** (G3 only; fixed before the freeze in `src/ensemble.py`: known customers → XGBoost, unknown →
CatBoost 0.6, LightGBM 0.2, XGBoost 0.1, random forest 0.1) needs no training. Once the four input models have their
G3 test rows, `python -m src.final_test --blend --freeze-confirmed` combines their saved test scores and applies the
blend's stored validation threshold.

A cell is shown only when all three seeds are in the log; section 1 lists what is missing.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src import ensemble, experiment, feature_sets as fs, final_test, segments
from src.metrics import top_rows
from src.split import holdout_indices

MODELS = list(final_test.TABLE_MODELS)          # six models plus the segment blend (G3 only)
BLEND = ensemble.BLEND_NAME
LABELS = final_test.NAMES
FEATURE_SETS = list(fs.FEATURE_SETS)
SEEDS = list(experiment.VAL_SEEDS)
N_BOOT = 200
METRICS = {"pr_auc": "PR-AUC", "auroc": "AUROC", "recall_top1pct": "recall in top 1%",
           "precision_top1pct": "precision in top 1%", "precision": "precision at stored threshold",
           "recall": "recall at stored threshold"}
pd.set_option("display.width", 160)

## 1. What has been run

In [2]:
spec = fs.load_spec()
ids = pd.read_parquet(ROOT / spec["parquet"],
                      columns=["TransactionID", "TransactionDT", "TransactionDay", "uid", "isFraud"])
train_idx, val_idx, test_idx = holdout_indices(ids)

log = experiment.current_runs(pd.read_csv(experiment.LOG_PATH))
test = log[(log["split"] == "test") & log["model"].isin(MODELS) & log["feature_set"].isin(FEATURE_SETS)].copy()
val = log[(log["split"] == "val") & (log["cv_scheme"] == "holdout") & log["model"].isin(MODELS)
          & log["feature_set"].isin(FEATURE_SETS)].copy()

seeds_done = test.groupby(["model", "feature_set"])["seed"].apply(lambda s: sorted(set(s)))
status = {(m, g): "n/a" if m == BLEND and g != ensemble.BLEND_SET else ("done" if seeds_done.get((m, g), []) == SEEDS else
                   f"{len(seeds_done.get((m, g), []))}/{len(SEEDS)} seeds" if (m, g) in seeds_done else "not run")
          for m in MODELS for g in FEATURE_SETS}
display(pd.DataFrame([[status[(m, g)] for g in FEATURE_SETS] for m in MODELS],
                     index=[LABELS[m] for m in MODELS], columns=FEATURE_SETS))
print(f"Test rows in the log: {len(test)} of {final_test.EXPECTED_TEST_ROWS}.")
# Nothing about the test block (labels, segment shares) is read until test rows exist, i.e. after the freeze.
FROZEN_RUN = len(test) > 0
n_test = len(test_idx)
fraud_test = int(ids["isFraud"].iloc[test_idx].sum()) if FROZEN_RUN else None
if FROZEN_RUN:
    print(f"test block: {n_test:,} rows, {fraud_test:,} fraud; the top 1% is {top_rows(n_test):,} rows.")
done = test[np.array([status[(m, g)] == "done" for m, g in zip(test["model"], test["feature_set"])], dtype=bool)].copy()
if FROZEN_RUN:
    done["precision_top1pct"] = done["recall_top1pct"] * fraud_test / top_rows(n_test)

,G0,G1,G2,G3
Logistic regression,not run,not run,not run,not run
Random forest,not run,not run,not run,not run
LightGBM,not run,not run,not run,not run
XGBoost,not run,not run,not run,not run
CatBoost,not run,not run,not run,not run
MLP,not run,not run,not run,not run
Segment blend (G3),n/a,n/a,n/a,not run


Test rows in the log: 0 of 75.


## 2. Test results (mean ± sample sd over seeds 0, 1, 2)

Precision in the top 1% is derived from the logged recall in the top 1%, as in `10_model_comparison`. Precision and
recall at the threshold use the threshold stored at validation, so unlike the validation tables they are not
optimistic.

In [3]:
def table(frame, metric):
    g = frame.groupby(["model", "feature_set"])[metric]
    cell = g.mean().map("{:.4f}".format) + " ± " + g.std().map("{:.4f}".format)
    out = cell.unstack().reindex(index=MODELS, columns=FEATURE_SETS)
    return out.rename(index=LABELS).fillna("—")

if done.empty:
    print("No complete test cells yet: run src.final_test for each model after the experiment freeze.")
else:
    for metric, label in METRICS.items():
        print(label)
        display(table(done, metric))

No complete test cells yet: run src.final_test for each model after the experiment freeze.


## 3. Validation to test

Mean PR-AUC on validation and on test for each cell, and the change. A drop from validation to test is expected
(later months, and validation was used to choose the settings and the threshold); what matters is whether the
ranking of models and feature sets holds.

In [4]:
if not done.empty:
    t_mean = done.groupby(["model", "feature_set"])["pr_auc"].mean()
    v_mean = val.groupby(["model", "feature_set"])["pr_auc"].mean()
    change = (t_mean - v_mean.reindex(t_mean.index)).unstack().reindex(index=MODELS, columns=FEATURE_SETS)
    print("change in mean PR-AUC, test − validation")
    display(change.rename(index=LABELS).round(4))
    rank = pd.DataFrame({"validation": v_mean.reindex(t_mean.index), "test": t_mean})
    rank["validation rank"] = rank["validation"].rank(ascending=False).astype(int)
    rank["test rank"] = rank["test"].rank(ascending=False).astype(int)
    print("cells ordered by test PR-AUC")
    display(rank.sort_values("test", ascending=False).rename(index=LABELS, level=0).round(4))

## 4. Known vs unknown customers (test block)

A test row is **known** if its `uid` appears in the train block, the rows every scored model was fitted on.
PR-AUC is the mean over seeds with a 95% bootstrap interval over rows (`N_BOOT` resamples), from the saved test
scores.

In [5]:
def known_unknown():
    known_by_id = pd.Series(segments.known_uid(ids["uid"], set(ids["uid"].iloc[train_idx])),
                            index=ids["TransactionID"].to_numpy())
    k_test = known_by_id.iloc[test_idx]
    print(f"test rows: known {k_test.mean():.1%}, unknown {1 - k_test.mean():.1%}")
    rows = []
    for m in MODELS:
        for g in FEATURE_SETS:
            paths = [ensemble.blend_preds_path(experiment.PREDS_DIR, "test", s) if m == BLEND
                 else final_test.preds_path(experiment.PREDS_DIR, m, g, s) for s in SEEDS]
            if status[(m, g)] != "done" or not all(p.exists() for p in paths):
                continue
            preds = [pd.read_parquet(p) for p in paths]
            if not all(p["TransactionID"].equals(preds[0]["TransactionID"]) for p in preds):
                raise ValueError(f"{m}/{g}: saved test scores cover different rows")
            known = known_by_id.reindex(preds[0]["TransactionID"]).to_numpy()
            t = segments.segment_pr_auc(preds[0]["isFraud"].to_numpy(), [p["score"].to_numpy() for p in preds],
                                        known, n_boot=N_BOOT)
            rows.append(t.assign(model=m, feature_set=g))
    if not rows:
        print("No saved test scores for complete cells yet.")
        return
    seg = pd.concat(rows, ignore_index=True)
    display(seg.groupby("segment")[["rows", "fraud"]].first())
    seg["cell"] = seg.apply(lambda r: f"{r.pr_auc:.4f} [{r.low:.3f}, {r.high:.3f}]", axis=1)
    display(seg.pivot_table(index="model", columns=["feature_set", "segment"], values="cell", aggfunc="first")
            .reindex(index=[m for m in MODELS if m in set(seg["model"])]).rename(index=LABELS))

if FROZEN_RUN:
    known_unknown()
else:
    print("No test rows yet; the test block is not read before the experiment freeze.")

No test rows yet; the test block is not read before the experiment freeze.


## 5. Segment blend vs XGBoost (G3)

The blend against its strongest input on the same feature set, seed by seed, on validation and test; and against
XGBoost on G1, the best single cell on validation. Known vs unknown for both is in section 4.

In [6]:
cmp = log[(log["split"].isin(["val", "test"])) & (log["cv_scheme"] == "holdout")
          & (((log["model"] == BLEND) & (log["feature_set"] == "G3"))
             | ((log["model"] == "xgboost") & log["feature_set"].isin(["G3", "G1"])))].copy()
cmp["cell"] = cmp["model"].map(LABELS) + " " + cmp["feature_set"]
if (cmp["split"] == "test").any():
    tab = cmp.pivot_table(index="cell", columns=["split", "seed"], values="pr_auc")
    tab[("test", "mean")] = tab["test"].mean(axis=1)
    tab[("val", "mean")] = tab["val"].mean(axis=1)
    display(tab.round(4))
else:
    print("No test rows yet.")

No test rows yet.


## 6. Findings

Write this after all 24 cells are done, quoting only numbers from the tables above:

- The best model and set on test, and its margin over the next best; whether the validation ranking held (RQ2).
- The gain between models against the gain from G0 to G3 on test, with G2 → G3 as the customer-profile step (RQ3).
- How much PR-AUC changed from validation to test, and whether some models or sets dropped more than others.
- Known vs unknown customers on test, and whether the gap matches validation.
- Precision and recall at the stored threshold: what share of fraud is caught, and how many flags are false.
- The segment blend against XGBoost on G3 and G1 (section 5): whether the validation gain held on test, and on which
  customers.